# Notebook Goal

This notebook builds and tests a reusable inference pipeline for the Credit Card Fraud Detection project.

Notebook `17_final_model_training.ipynb` already trained and saved the final validated fraud model and its supporting artifacts. Notebook `18_inference_pipeline.ipynb` will load those saved artifacts and use them to make predictions.

No model training happens in this notebook. No threshold tuning happens in this notebook. The selected features and saved decision policy from notebook 17 are reused as-is.

The purpose of this notebook is to simulate how the fraud model will work in production: load the saved model, prepare input data in the expected format, generate a fraud probability, and convert that result into a reusable prediction workflow.

This notebook also prepares the project for the next FastAPI `/predict` endpoint by turning the saved model artifacts into a clear, testable inference path.


# Load Saved Artifacts

This section loads the final validated model and the supporting JSON artifacts created in notebook `17_final_model_training.ipynb`.

The goal here is only to verify that the saved inference assets exist and can be loaded correctly before later steps use them for prediction.


In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

FINAL_MODEL_PATH = ARTIFACTS_DIR / "final_validated_fraud_model.joblib"
FINAL_FEATURE_COLUMNS_PATH = ARTIFACTS_DIR / "final_feature_columns.json"
FINAL_DECISION_POLICY_PATH = ARTIFACTS_DIR / "final_decision_policy.json"
FINAL_MODEL_METADATA_PATH = ARTIFACTS_DIR / "final_model_metadata.json"
FINAL_MODEL_METRICS_PATH = ARTIFACTS_DIR / "final_model_metrics.json"

artifact_paths = {
    "final_validated_model": FINAL_MODEL_PATH,
    "final_feature_columns": FINAL_FEATURE_COLUMNS_PATH,
    "final_decision_policy": FINAL_DECISION_POLICY_PATH,
    "final_model_metadata": FINAL_MODEL_METADATA_PATH,
    "final_model_metrics": FINAL_MODEL_METRICS_PATH,
}

for artifact_name, artifact_path in artifact_paths.items():
    if not artifact_path.exists():
        raise FileNotFoundError(f"Missing required artifact: {artifact_path}")

    print(f"Found {artifact_name}: {artifact_path}")


Found final_validated_model: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_validated_fraud_model.joblib
Found final_feature_columns: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_feature_columns.json
Found final_decision_policy: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_decision_policy.json
Found final_model_metadata: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_model_metadata.json
Found final_model_metrics: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_model_metrics.json


In [3]:
final_model = joblib.load(FINAL_MODEL_PATH)
print(f"Loaded final validated model successfully: {FINAL_MODEL_PATH}")

with open(FINAL_FEATURE_COLUMNS_PATH, "r", encoding="utf-8") as f:
    final_feature_columns = json.load(f)
print(f"Loaded final feature columns successfully: {FINAL_FEATURE_COLUMNS_PATH}")

with open(FINAL_DECISION_POLICY_PATH, "r", encoding="utf-8") as f:
    final_decision_policy = json.load(f)
print(f"Loaded final decision policy successfully: {FINAL_DECISION_POLICY_PATH}")

with open(FINAL_MODEL_METADATA_PATH, "r", encoding="utf-8") as f:
    final_model_metadata = json.load(f)
print(f"Loaded final model metadata successfully: {FINAL_MODEL_METADATA_PATH}")

with open(FINAL_MODEL_METRICS_PATH, "r", encoding="utf-8") as f:
    final_model_metrics = json.load(f)
print(f"Loaded final model metrics successfully: {FINAL_MODEL_METRICS_PATH}")


Loaded final validated model successfully: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_validated_fraud_model.joblib
Loaded final feature columns successfully: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_feature_columns.json
Loaded final decision policy successfully: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_decision_policy.json
Loaded final model metadata successfully: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_model_metadata.json
Loaded final model metrics successfully: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_model_metrics.json
